# AIRL Time/Zonal Daily Comparison

This notebook evaluates four trained AIRL policies on the **same validation split** used in the time-aware training notebooks.

Checkpoint selection:
- Set `USE_FINAL_POLICY_STATE` / `USE_FINAL_REWARD_STATE` to `True` to load the saved `final_state` files.
- Leave them `False` to use the explicit 0-based iteration indices.

Outputs:
- One figure per validation day (`2x2` subplots) comparing daily trajectories across the four AIRL configurations.
- One grouped bar chart comparing daily HVAC totals of the four AIRL settings against historical data.


In [ ]:
import os
import re
import sys
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


def _iter_project_candidates(start: Path):
    seen = set()

    first_pass = [start]
    try:
        first_pass.extend(child for child in start.iterdir() if child.is_dir())
    except OSError:
        pass

    for candidate in first_pass:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        yield candidate

    for candidate in start.parents:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        yield candidate


def _locate_project_root(start: Path) -> Path:
    candidates = []
    for candidate in _iter_project_candidates(start):
        try:
            has_layout = (candidate / 'src').is_dir() and (candidate / 'configs').is_dir()
        except OSError:
            has_layout = False
        if not has_layout:
            continue

        score = 0
        if (candidate / 'notebooks' / 'README.md').is_file():
            score += 2
        if (candidate / 'tests' / 'test_standalone_repo.py').is_file():
            score += 2
        candidates.append((score, candidate))

    if not candidates:
        raise RuntimeError('Unable to locate project root from ' + str(start))

    best_score, best_candidate = candidates[0]
    for score, candidate in candidates[1:]:
        if score > best_score:
            best_score, best_candidate = score, candidate
    return best_candidate


try:
    NOTEBOOK_PATH = Path(__file__).resolve()
    NOTEBOOK_DIR = NOTEBOOK_PATH.parent
except NameError:
    NOTEBOOK_PATH = None
    NOTEBOOK_DIR = Path.cwd().resolve()

PROJECT_ROOT = _locate_project_root(NOTEBOOK_DIR)
if NOTEBOOK_DIR.name != 'notebooks' and (PROJECT_ROOT / 'notebooks').is_dir():
    NOTEBOOK_DIR = PROJECT_ROOT / 'notebooks'

if Path.cwd().resolve() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

src_path = str(PROJECT_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

plt.style.use('seaborn-v0_8')
print(f'Notebook path: {NOTEBOOK_PATH}')
print(f'Notebook directory: {NOTEBOOK_DIR}')
print(f'Project root: {PROJECT_ROOT}')
print(f'Using torch {torch.__version__}')

from configs.training_config import get_full_training_config
from data_processing import (
    load_and_filter_data,
    split_train_val,
    setup_scalers,
    set_random_seed,
    inverse_normalize,
)
from models import load_dynamics_model, MMVPolicyActorCritic
from airl_environment import create_airl_environment


In [ ]:
# Mirror the dataset/split setup from airl_mode_control_training_temp_win_energy_consistent*.ipynb
config = get_full_training_config()
config.data.data_file = 'l14_merged_data_with_rain.csv'
config.data.min_trajectory_length = 690
config.environment.comfort_max = 27.0  # same training comfort max in source notebooks
REPORT_COMFORT_CEILING = 30.0          # same reporting comfort ceiling in source notebooks

set_random_seed(config.data.random_seed)
device = torch.device('cpu')
print(f'Device: {device}')
print(config)


data_path = PROJECT_ROOT / config.data.data_file
print(f'Loading data from {data_path}')

daily_dataframes = load_and_filter_data(
    file_path=str(data_path),
    min_length=config.data.min_trajectory_length,
)
print(f'Total usable days before exclusions: {len(daily_dataframes)}')

# Keep the same exclusions used in the four source training notebooks.
dates_to_exclude = {(7, 17), (8, 9)}
filtered_daily = []
removed_labels = []
for df in daily_dataframes:
    day = pd.to_datetime(df['date'].iloc[0])
    if (day.month, day.day) in dates_to_exclude:
        removed_labels.append(day.strftime('%Y-%m-%d'))
        continue
    filtered_daily.append(df)

daily_dataframes = filtered_daily
print(f'Removed {len(removed_labels)} excluded day(s): {removed_labels}')
print(f'Total usable days after exclusions: {len(daily_dataframes)}')

train_dfs, val_dfs = split_train_val(
    daily_dataframes,
    split_ratio=config.data.train_val_split,
    shuffle=True,
    random_state=0,
)
print(f'Train days: {len(train_dfs)} | Validation days: {len(val_dfs)}')

scalers = setup_scalers(train_dfs, val_dfs)
print(f'Created scalers for {len(scalers)} features')


In [ ]:
# Load dynamics model and construct base environment kwargs for validation replay
model_path = PROJECT_ROOT / config.dynamics_model_path
print(f'Loading dynamics model from {model_path}')

dynamics_model = load_dynamics_model(
    model_path=str(model_path),
    device=torch.device('cpu'),
    hidden_dim=config.model.dynamics_hidden_dim,
    output_dim=config.model.dynamics_output_dim,
    ac_dim=config.model.dynamics_ac_channels,
    nv_dim=config.model.dynamics_nv_channels,
)
dynamics_model.eval()

report_env_kwargs = dict(
    daily_data_list=val_dfs,
    dynamics_model=dynamics_model,
    scalers=scalers,
    num_zones=config.model.num_zones,
    comfort_min=config.environment.comfort_min,
    comfort_max=REPORT_COMFORT_CEILING,
    heavy_wind_threshold=config.environment.heavy_wind_threshold,
    policy_time_features=True,
)

base_env = create_airl_environment(**report_env_kwargs)
initial_state = base_env.reset(day_index=0)
state_dim = initial_state.shape[0]
zone_cols = base_env.columns['zone_cols'][:config.model.num_zones]
outdoor_col = 'OutdoorTemperatureWindow'

print(f'State dimension: {state_dim}')
print(f'Num zones used: {len(zone_cols)}')


In [ ]:
# Four time-aware AIRL configurations from the *_time*_gru notebooks.
USE_FINAL_POLICY_STATE = True
USE_FINAL_REWARD_STATE = True
POLICY_CHECKPOINT_INDEX = 19
REWARD_CHECKPOINT_INDEX = 19

checkpoint_specs: List[Dict[str, str]] = [
    {
        'key': 'time_only',
        'label': 'State Only',
        'reward_type': 'temps_hold_dwell_prev_time_gru',
        'line_color': 'tab:blue',
    },
    {
        'key': 'time_comfort_zonal',
        'label': 'State + Comfort',
        'reward_type': 'temps_hold_dwell_prev_time_comfort_zonal_gru',
        'line_color': 'tab:green',
    },
    {
        'key': 'time_energy_zonal',
        'label': 'State + Energy',
        'reward_type': 'temps_hold_dwell_prev_time_energy_zonal_gru',
        'line_color': 'tab:purple',
    },
    {
        'key': 'time_energy_comfort_zonal',
        'label': 'State + Energy + Comfort',
        'reward_type': 'temps_hold_dwell_prev_time_energy_comfort_zonal_gru',
        'line_color': 'tab:red',
    },
]

checkpoints_root = PROJECT_ROOT / 'notebooks/outputs'


def _selection_label(use_final_state: bool, checkpoint_index: int) -> str:
    return 'final_state' if use_final_state else f'iter_{checkpoint_index:02d}'


POLICY_SELECTION_LABEL = _selection_label(USE_FINAL_POLICY_STATE, POLICY_CHECKPOINT_INDEX)
REWARD_SELECTION_LABEL = _selection_label(USE_FINAL_REWARD_STATE, REWARD_CHECKPOINT_INDEX)
COMPARISON_TAG = f'policy_{POLICY_SELECTION_LABEL}_reward_{REWARD_SELECTION_LABEL}'
print(f'Policy selection: {POLICY_SELECTION_LABEL}')
print(f'Reward selection: {REWARD_SELECTION_LABEL}')


def _resolve_checkpoint(reward_type: str, prefix: str, checkpoint_index: int, use_final_state: bool) -> Path:
    if use_final_state:
        if prefix == 'AIRL_policy':
            filename = f'airl_mode_policy_{reward_type}_final_state.pth'
        elif prefix == 'AIRL_reward':
            filename = f'airl_mode_reward_{reward_type}_final_state.pth'
        else:
            raise ValueError(f'Unsupported checkpoint prefix: {prefix}')
    else:
        filename = f'{prefix}_{reward_type}_{checkpoint_index}.pth'

    path = checkpoints_root / filename
    if not path.exists():
        if use_final_state:
            raise FileNotFoundError(
                f"Missing final-state checkpoint for reward_type='{reward_type}', prefix='{prefix}': {path}. "
                "Create the corresponding *_final_state.pth file first, or set the USE_FINAL_*_STATE flag back to False."
            )
        raise FileNotFoundError(
            f"Missing checkpoint for reward_type='{reward_type}', prefix='{prefix}': {path}"
        )
    return path


policies: Dict[str, MMVPolicyActorCritic] = {}
for spec in checkpoint_specs:
    policy_ckpt = _resolve_checkpoint(
        spec['reward_type'],
        'AIRL_policy',
        POLICY_CHECKPOINT_INDEX,
        USE_FINAL_POLICY_STATE,
    )
    reward_ckpt = _resolve_checkpoint(
        spec['reward_type'],
        'AIRL_reward',
        REWARD_CHECKPOINT_INDEX,
        USE_FINAL_REWARD_STATE,
    )
    spec['policy_checkpoint'] = str(policy_ckpt.relative_to(PROJECT_ROOT))
    spec['reward_checkpoint'] = str(reward_ckpt.relative_to(PROJECT_ROOT))

    policy = MMVPolicyActorCritic(state_dim=state_dim, num_zones=config.model.num_zones).to(device)
    state_dict = torch.load(policy_ckpt, map_location=device)

    # Handle possible DataParallel prefixes.
    if isinstance(state_dict, dict) and any(str(k).startswith('module.') for k in state_dict.keys()):
        state_dict = {str(k).replace('module.', '', 1): v for k, v in state_dict.items()}

    policy.load_state_dict(state_dict)
    policy.eval()
    policies[spec['key']] = policy
    print(
        f"Loaded {spec['label']:<32} <- policy {spec['policy_checkpoint']} | "
        f"reward {spec['reward_checkpoint']}"
    )


In [ ]:
def _make_eval_env():
    return create_airl_environment(**report_env_kwargs)


def _rollout_day(env, policy: MMVPolicyActorCritic, day_index: int) -> Dict[str, np.ndarray]:
    state = env.reset(day_index=day_index)
    done = False

    avg_zone_trace: List[float] = []
    outdoor_trace: List[float] = []
    window_trace: List[int] = []
    energy_trace: List[float] = []
    avg_fcu_input_trace: List[float] = []
    avg_pfcu_input_trace: List[float] = []

    while not done:
        state_tensor = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            action_dict, _ = policy.get_action(state_tensor)

        next_state, _, done, info = env.step(action_dict)

        zone_c = [
            inverse_normalize(env.zone_temps[i], env.columns['zone_cols'][i], env.scalers)
            for i in range(config.model.num_zones)
        ]
        avg_zone_c = float(np.mean(zone_c))
        avg_zone_trace.append(avg_zone_c)

        outside_val = info.get('outdoor_temp_c')
        if outside_val is None:
            outside_idx = len(env.zone_temps)
            if next_state.shape[0] > outside_idx:
                outside_val = inverse_normalize(float(next_state[outside_idx]), outdoor_col, env.scalers)
            elif hasattr(env, 'outdoor_temp'):
                outside_val = float(env.outdoor_temp)
        outdoor_c = float(outside_val) if outside_val is not None else np.nan
        outdoor_trace.append(outdoor_c)

        window_state = int(info.get('window_state', info.get('window', getattr(env, 'window_prev_state', getattr(env, 'prev_window', 0)))))
        window_trace.append(window_state)
        energy_trace.append(float(info.get('energy_consumption', info.get('energy', 0.0))))

        fcu_cols = env.columns['fcu_supply_cols'][:config.model.num_zones]
        raw_supply = np.asarray(getattr(env, 'prev_supply_temps', []), dtype=np.float32).reshape(-1)
        fcu_inputs = []
        for zone_idx, zone_temp_c in enumerate(zone_c):
            if window_state == 0 and zone_idx < raw_supply.size and zone_idx < len(fcu_cols):
                fcu_temp_c = inverse_normalize(float(raw_supply[zone_idx]), fcu_cols[zone_idx], env.scalers)
            else:
                fcu_temp_c = float(zone_temp_c)
            fcu_inputs.append(float(fcu_temp_c))
        avg_fcu_input_trace.append(float(np.mean(fcu_inputs)) if fcu_inputs else avg_zone_c)

        lc_cols = env.columns['lc_cols']
        raw_pfcu = np.asarray(getattr(env, 'prev_lc_temps', []), dtype=np.float32).reshape(-1)
        pfcu_active = int(info.get('doas', 0)) == 1

        pfcu_inputs = []
        for lc_idx, col in enumerate(lc_cols):
            if pfcu_active and lc_idx < raw_pfcu.size:
                pfcu_temp_c = inverse_normalize(float(raw_pfcu[lc_idx]), col, env.scalers)
            else:
                pfcu_temp_c = outdoor_c
            pfcu_inputs.append(float(pfcu_temp_c))
        avg_pfcu_input_trace.append(float(np.mean(pfcu_inputs)) if pfcu_inputs else outdoor_c)

        state = next_state

    return {
        'avg_zone_temp': np.asarray(avg_zone_trace, dtype=np.float32),
        'outdoor_temp': np.asarray(outdoor_trace, dtype=np.float32),
        'window_state': np.asarray(window_trace, dtype=np.int32),
        'energy_kwh_per_min': np.asarray(energy_trace, dtype=np.float32),
        'avg_fcu_input_temp': np.asarray(avg_fcu_input_trace, dtype=np.float32),
        'avg_pfcu_input_temp': np.asarray(avg_pfcu_input_trace, dtype=np.float32),
        'total_energy_kwh': float(np.sum(energy_trace)),
    }


def _compute_historical_energy_kwh(day_df: pd.DataFrame) -> Tuple[float, str]:
    # Preferred: direct total HVAC power channel
    zone_total_cols = [c for c in day_df.columns if c.strip().lower() == 'zone total power watt']
    if zone_total_cols:
        watts = pd.to_numeric(day_df[zone_total_cols[0]], errors='coerce').fillna(0.0).to_numpy(dtype=float)
        watts = np.clip(watts, 0.0, None)
        return float(np.sum(watts) / 1000.0 / 60.0), f"{zone_total_cols[0]}"

    # Fallback: sum all FCU/PFCU watt channels
    watt_cols = [
        c for c in day_df.columns
        if c.endswith(' Watt') and (c.startswith('FCU-') or c.startswith('PFCU-'))
    ]
    if watt_cols:
        watts = day_df[watt_cols].apply(pd.to_numeric, errors='coerce').fillna(0.0).to_numpy(dtype=float)
        watts = np.clip(watts, 0.0, None)
        return float(np.sum(watts) / 1000.0 / 60.0), f"sum({len(watt_cols)} FCU/PFCU watt cols)"

    # Last-resort fallback if only energy-like channel exists
    if 'Zone Total Power KWh' in day_df.columns:
        kwh = pd.to_numeric(day_df['Zone Total Power KWh'], errors='coerce').fillna(0.0).to_numpy(dtype=float)
        kwh = np.clip(kwh, 0.0, None)
        return float(np.sum(kwh)), 'Zone Total Power KWh'

    return float('nan'), 'missing power columns'


# Run all four policies over all validation days
all_results: Dict[str, Dict[str, object]] = {}
num_val_days = len(val_dfs)
for spec in checkpoint_specs:
    env = _make_eval_env()
    traces = []
    rows = []

    for day_idx in range(num_val_days):
        day_trace = _rollout_day(env, policies[spec['key']], day_idx)
        traces.append(day_trace)
        rows.append({
            'day_index': day_idx,
            'total_energy_kwh': day_trace['total_energy_kwh'],
        })

    metrics_df = pd.DataFrame(rows)
    all_results[spec['key']] = {
        'spec': spec,
        'traces': traces,
        'metrics': metrics_df,
    }
    print(f"{spec['label']:<30} | simulated days: {len(traces)} | mean total kWh/day: {metrics_df['total_energy_kwh'].mean():.3f}")


historical_rows = []
historical_notes = set()
for day_idx, day_df in enumerate(val_dfs):
    total_kwh, source_note = _compute_historical_energy_kwh(day_df)
    historical_rows.append({
        'day_index': day_idx,
        'historical_total_energy_kwh': total_kwh,
    })
    historical_notes.add(source_note)

historical_df = pd.DataFrame(historical_rows)
print(f'Historical energy source note(s): {sorted(historical_notes)}')
display(historical_df.head())


In [ ]:
# Daily 2x2 trajectory comparison figures (one figure per validation day)
output_dir = PROJECT_ROOT / f'notebooks/outputs/four_airl_time_zonal_daily_compare_{COMPARISON_TAG}'
output_dir.mkdir(parents=True, exist_ok=True)


def _shade_window_state(ax: plt.Axes, window_seq: np.ndarray) -> None:
    if window_seq.size == 0:
        return
    change_points = np.where(np.diff(window_seq) != 0)[0] + 1
    starts = np.concatenate(([0], change_points))
    ends = np.concatenate((change_points, [len(window_seq)]))
    for start, end in zip(starts, ends):
        status = int(window_seq[start])
        shade_color = '#d9d9d9' if status == 0 else '#8fb8de'
        shade_alpha = 0.18 if status == 0 else 0.35
        ax.axvspan(start, end, color=shade_color, alpha=shade_alpha, zorder=0)


def _half_hour_ticks(day_df: pd.DataFrame, series_length: int) -> tuple[np.ndarray, list[str]]:
    timestamps = pd.to_datetime(day_df['date']).reset_index(drop=True)
    usable_len = min(series_length, len(timestamps))
    if usable_len <= 0:
        return np.asarray([], dtype=int), []

    tick_positions = list(range(0, usable_len, 30))
    if tick_positions[-1] != usable_len - 1:
        tick_positions.append(usable_len - 1)

    tick_labels = [timestamps.iloc[pos].strftime('%H:%M') for pos in tick_positions]
    return np.asarray(tick_positions, dtype=int), tick_labels


saved_daily_paths = []
for day_idx in range(num_val_days):
    fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True, sharey=True)
    axes_flat = axes.flatten()
    tick_positions, tick_labels = _half_hour_ticks(val_dfs[day_idx], len(all_results[checkpoint_specs[0]['key']]['traces'][day_idx]['avg_zone_temp']))

    for ax, spec in zip(axes_flat, checkpoint_specs):
        trace = all_results[spec['key']]['traces'][day_idx]
        avg_zone = trace['avg_zone_temp']
        outdoor = trace['outdoor_temp']
        window = trace['window_state']

        minutes = np.arange(len(avg_zone))
        _shade_window_state(ax, window)
        ax.plot(minutes, outdoor, color='tab:orange', linewidth=1.6, zorder=2.5)
        ax.axhline(REPORT_COMFORT_CEILING, color='tab:red', linestyle='--', linewidth=1.2, zorder=2)
        ax.plot(minutes, avg_zone, color=spec['line_color'], linewidth=2.0, zorder=3)

        ax.set_title(spec['label'])
        ax.set_ylabel('Temperature (°C)')
        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_labels, rotation=45, ha='right')
        ax.grid(True, linestyle='--', alpha=0.3)

    axes_flat[-2].set_xlabel('Time (HH:MM)')
    axes_flat[-1].set_xlabel('Time (HH:MM)')

    legend_handles = [
        Line2D([0], [0], color='tab:orange', linewidth=1.6, label='Outdoor temp (°C)'),
        Line2D([0], [0], color='tab:red', linestyle='--', linewidth=1.2, label=f'Comfort limit ({REPORT_COMFORT_CEILING:.0f}°C)'),
        Patch(facecolor='#d9d9d9', edgecolor='none', alpha=0.18, label='Window closed'),
        Patch(facecolor='#8fb8de', edgecolor='none', alpha=0.35, label='Window open'),
    ]
    fig.legend(handles=legend_handles, loc='lower center', ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.01))
    fig.suptitle(
        f'Validation day {day_idx}: daily trajectory comparison across four time-aware AIRL settings '
        f'({POLICY_SELECTION_LABEL}, {REWARD_SELECTION_LABEL})',
        fontsize=14,
    )
    fig.tight_layout(rect=(0.0, 0.05, 1.0, 0.95))

    fig_path = output_dir / f'day_{day_idx:02d}_four_airl_time_zonal_compare_{COMPARISON_TAG}.png'
    fig.savefig(fig_path, dpi=160, bbox_inches='tight')
    plt.close(fig)
    saved_daily_paths.append(fig_path)

print(f'Saved {len(saved_daily_paths)} daily comparison figures to: {output_dir}')
print('First 3 files:')
for p in saved_daily_paths[:3]:
    print(' -', p)


In [ ]:
# Daily 2x2 comparison figures with FCU/PFCU input panels for each configuration
detailed_output_dir = PROJECT_ROOT / f'notebooks/outputs/four_airl_time_zonal_daily_compare_inputs_{COMPARISON_TAG}'
detailed_output_dir.mkdir(parents=True, exist_ok=True)

set2 = plt.get_cmap('Set2').colors
color_outdoor = set2[1]
color_comfort = set2[0]
color_fcu = set2[2]
color_pfcu = set2[3]
color_window_closed = set2[7]
color_window_open = set2[4]
panel_colors = [checkpoint_specs[i]['line_color'] for i in range(len(checkpoint_specs))]

saved_detailed_paths = []
for day_idx in range(num_val_days):
    with plt.style.context('seaborn-v0_8-whitegrid'):
        fig = plt.figure(figsize=(18, 14))
    fig = plt.figure(figsize=(18, 14))
    # fig.patch.set_facecolor('#EAEAF2')
    outer = fig.add_gridspec(2, 2, wspace=0.16, hspace=0.18)
    base_trace = all_results[checkpoint_specs[0]['key']]['traces'][day_idx]
    tick_positions, tick_labels = _half_hour_ticks(val_dfs[day_idx], len(base_trace['avg_zone_temp']))

    for panel_idx, spec in enumerate(checkpoint_specs):
        outer_row, outer_col = divmod(panel_idx, 2)
        inner = outer[outer_row, outer_col].subgridspec(2, 1, height_ratios=[2.0, 1.15], hspace=0.08)
        ax_top = fig.add_subplot(inner[0])
        ax_bottom = fig.add_subplot(inner[1], sharex=ax_top)

        for ax in [ax_top, ax_bottom]:
            # ax.set_facecolor('#EAEAF2')
            ax.grid(False)

        trace = all_results[spec['key']]['traces'][day_idx]
        avg_zone = trace['avg_zone_temp']
        outdoor = trace['outdoor_temp']
        window = trace['window_state']
        avg_fcu_input = trace['avg_fcu_input_temp']
        avg_pfcu_input = trace['avg_pfcu_input_temp']
        minutes = np.arange(len(avg_zone))

        _shade_window_state(ax_top, window)
        ax_top.plot(minutes, outdoor, color=color_outdoor, linewidth=1.5, zorder=2.5)
        ax_top.axhline(REPORT_COMFORT_CEILING, color=color_comfort, linestyle='--', linewidth=1.1, zorder=2)
        ax_top.plot(minutes, avg_zone, color=panel_colors[panel_idx], linewidth=2.0, zorder=3)
        ax_top.set_title(spec['label'])
        ax_top.set_ylabel('Temp (°C)')
        ax_top.tick_params(axis='x', labelbottom=False)

        _shade_window_state(ax_bottom, window)
        ax_bottom.plot(minutes, avg_fcu_input, color=color_fcu, linewidth=1.9, zorder=3)
        ax_bottom.plot(minutes, avg_pfcu_input, color=color_pfcu, linewidth=1.9, zorder=3)
        ax_bottom.set_ylabel('Input (°C)')
        ax_bottom.set_xlabel('Time (HH:MM)')
        ax_bottom.set_xticks(tick_positions)
        ax_bottom.set_xticklabels(tick_labels, rotation=45, ha='right')

    detailed_legend_handles = [
        Line2D([0], [0], color=color_outdoor, linewidth=1.5, label='Outdoor temp (°C)'),
        Line2D([0], [0], color=color_comfort, linestyle='--', linewidth=1.1, label=f'Comfort limit ({REPORT_COMFORT_CEILING:.0f}°C)'),
        Line2D([0], [0], color=color_fcu, linewidth=1.9, label='Avg FCU input'),
        Line2D([0], [0], color=color_pfcu, linewidth=1.9, label='Avg DOAS input'),
        Patch(facecolor=color_window_closed, edgecolor='none', alpha=0.18, label='Window closed'),
        Patch(facecolor=color_window_open, edgecolor='none', alpha=0.35, label='Window open'),
    ]
    fig.legend(handles=detailed_legend_handles, loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.01))
    fig.tight_layout(rect=(0.0, 0.05, 1.0, 0.96))

    fig_path = detailed_output_dir / f'day_{day_idx:02d}_four_airl_time_zonal_compare_inputs_{COMPARISON_TAG}.png'
    fig.savefig(fig_path, dpi=160, bbox_inches='tight')
    plt.close(fig)
    saved_detailed_paths.append(fig_path)

print(f'Saved {len(saved_detailed_paths)} detailed daily comparison figures to: {detailed_output_dir}')
print('First 3 files:')
for p in saved_detailed_paths[:3]:
    print(' -', p)


In [ ]:
# Grouped bar plot: all validation days, 5 bars per day (4 AIRL + historical)
bar_df = pd.DataFrame({'day_index': np.arange(num_val_days, dtype=int)})
for spec in checkpoint_specs:
    metric_col = f"{spec['key']}_total_energy_kwh"
    mdf = all_results[spec['key']]['metrics'][['day_index', 'total_energy_kwh']].rename(
        columns={'total_energy_kwh': metric_col}
    )
    bar_df = bar_df.merge(mdf, on='day_index', how='left')

bar_df = bar_df.merge(historical_df, on='day_index', how='left')

fig, ax = plt.subplots(figsize=(16, 6))
x = np.arange(num_val_days)
width = 0.16

series_order = [
    (checkpoint_specs[0]['label'], f"{checkpoint_specs[0]['key']}_total_energy_kwh", checkpoint_specs[0]['line_color']),
    (checkpoint_specs[1]['label'], f"{checkpoint_specs[1]['key']}_total_energy_kwh", checkpoint_specs[1]['line_color']),
    (checkpoint_specs[2]['label'], f"{checkpoint_specs[2]['key']}_total_energy_kwh", checkpoint_specs[2]['line_color']),
    (checkpoint_specs[3]['label'], f"{checkpoint_specs[3]['key']}_total_energy_kwh", checkpoint_specs[3]['line_color']),
    ('Historical (metered)', 'historical_total_energy_kwh', 'tab:gray'),
]

offsets = np.linspace(-2 * width, 2 * width, len(series_order))
for offset, (label, col, color) in zip(offsets, series_order):
    ax.bar(x + offset, bar_df[col].to_numpy(dtype=float), width=width, label=label, color=color, alpha=0.9)

ax.set_xticks(x)
ax.set_xticklabels([f'Day {d}' for d in bar_df['day_index'].to_numpy(dtype=int)], rotation=45)
ax.set_ylabel('Daily HVAC energy (kWh/day)')
ax.set_title(
    'Validation daily HVAC totals: four time-aware AIRL settings vs historical data '
    f'({POLICY_SELECTION_LABEL}, {REWARD_SELECTION_LABEL})'
)
ax.grid(axis='y', linestyle='--', alpha=0.3)
ax.legend(loc='upper right', ncol=2, frameon=False)
fig.tight_layout()

bar_path = output_dir / f'validation_daily_hvac_energy_bar_{COMPARISON_TAG}.png'
fig.savefig(bar_path, dpi=170, bbox_inches='tight')
plt.show()

print(f'Saved bar plot to: {bar_path}')
display(bar_df)


In [ ]:
# Quick file inventory
from pathlib import Path
inventory_dir = PROJECT_ROOT / f'notebooks/outputs/four_airl_time_zonal_daily_compare_{COMPARISON_TAG}'
all_pngs = sorted(inventory_dir.glob('*.png'))
print(f'Total generated PNG files: {len(all_pngs)}')
for p in all_pngs[:5]:
    print(' -', p)
